In [1]:
import os
import math
import time
from itertools import groupby

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
ALPHABETS = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ "

MAX_STR_LEN = 19

NUM_CHARACTERS = len(ALPHABETS) + 1

IMG_H = 32
IMG_W = 128

BATCH_SIZE = 16

EPOCHS = 150

LR = 0.001

DATASET_PATH = "../Datasets/iam_words/"

In [4]:
def label_to_num(txt):

    encoded = []

    for ch in txt:
        try:
            encoded.append(ALPHABETS.index(ch))
        except ValueError:
            pass

    encoded = encoded[:MAX_STR_LEN]

    encoded += [len(ALPHABETS)] * (MAX_STR_LEN - len(encoded))

    return encoded


def num_to_label(num):

    text = ""

    for ch in num:

        if ch == -1:
            break

        if ch < len(ALPHABETS):
            text += ALPHABETS[ch]

    return text

In [5]:
def ctc_decoder(predictions):

    pred_indices = np.argmax(predictions, axis=2)

    text_list = []

    for row in pred_indices:

        merged = [k for k, _ in groupby(row)]

        text = ""

        for p in merged:

            if p != len(ALPHABETS):
                text += ALPHABETS[int(p)]

        text_list.append(text)

    return text_list

In [6]:
def load_iam_dataset(data_root):

    words_txt = os.path.join(data_root, "words.txt")

    samples = []

    with open(words_txt, "r", encoding="utf-8") as f:

        for line in f:

            if line.startswith("#"):
                continue

            parts = line.strip().split()

            if len(parts) < 9:
                continue

            word_id = parts[0]
            status = parts[1]

            if status != "ok":
                continue

            label = " ".join(parts[8:])

            folder1 = word_id.split("-")[0]
            folder2 = "-".join(word_id.split("-")[:2])

            img_path = os.path.join(
                data_root,
                "words",
                folder1,
                folder2,
                f"{word_id}.png"
            )

            if os.path.exists(img_path):
                samples.append((img_path, label))

    return samples

In [7]:
samples = load_iam_dataset(DATASET_PATH)

df = pd.DataFrame(samples, columns=["Fpath", "Identify"])

train_df = df.sample(frac=0.9, random_state=42)

valid_df = df.drop(train_df.index)

train_df = train_df.reset_index(drop=True)

valid_df = valid_df.reset_index(drop=True)

print(len(train_df))
print(len(valid_df))

34474
3831


In [8]:
class IAMDataset(Dataset):

    def __init__(self, df):

        self.paths = df["Fpath"].tolist()

        self.labels = df["Identify"].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        img = cv2.imread(
            self.paths[idx],
            cv2.IMREAD_GRAYSCALE
        )

        img = cv2.resize(
            img,
            (IMG_W, IMG_H)
        )

        img = img.astype(np.float32) / 255.0

        img = torch.tensor(
            img,
            dtype=torch.float32
        ).unsqueeze(0)

        label = torch.tensor(
            label_to_num(self.labels[idx]),
            dtype=torch.long
        )

        return img, label

In [9]:
train_dataset = IAMDataset(train_df)

valid_dataset = IAMDataset(valid_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train Batches:", len(train_loader))

Train Batches: 2155


In [11]:
class CRNN(nn.Module):

    def __init__(self, num_characters: int = NUM_CHARACTERS):
        super().__init__()

        # --- CNN backbone ---
        self.cnn = nn.Sequential(
            # conv1 + pool
            nn.Conv2d(1,   32,  3, padding=1), nn.SELU(),
            nn.MaxPool2d((2, 2)),

            # conv2 + pool
            nn.Conv2d(32,  64,  3, padding=1), nn.SELU(),
            nn.MaxPool2d((2, 2)),

            # conv3, conv4
            nn.Conv2d(64,  128, 3, padding=1), nn.SELU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.SELU(),

            # conv5, conv6 + dropout
            nn.Conv2d(128, 512, 3, padding=1), nn.SELU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.SELU(),
            nn.Dropout2d(0.2),

            # conv7, conv8 + pool (2×1 — height halved, width kept)
            nn.Conv2d(512, 512, 3, padding=1), nn.SELU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.SELU(),
            nn.MaxPool2d((2, 1)),

            # conv9 + BN + dropout
            nn.Conv2d(512, 256, 3, padding=1), nn.SELU(),
            nn.BatchNorm2d(256),
            nn.Dropout2d(0.2),

            # conv10 + BN + pool (2×1) + dropout
            nn.Conv2d(256, 256, 3, padding=1), nn.SELU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d((2, 1)),
            nn.Dropout2d(0.2),

            # conv11 (2×2, NO padding → height collapses to 1) + dropout
            nn.Conv2d(256, 64, (2, 2)), nn.SELU(),
            nn.Dropout2d(0.2),
        )

        # --- Sequence model (BiLSTM stack) ---
        # After CNN the spatial map is (N, 64, 1, W')
        # We squeeze the height dim → (N, W', 64) as the time-series input.
        self.lstm1 = nn.LSTM(64,  128, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(256, 512, batch_first=True, bidirectional=True)
        self.lstm3 = nn.LSTM(1024,512, batch_first=True, bidirectional=True)
        self.lstm4 = nn.LSTM(1024,512, batch_first=True, bidirectional=True)
        self.lstm5 = nn.LSTM(1024,128, batch_first=True, bidirectional=True)

        # --- Output head ---
        self.dense1 = nn.Linear(256, 128)
        self.dense2 = nn.Linear(128, num_characters)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (N, 1, H, W)  float in [0, 1]
        Returns:
            out : (T, N, num_characters)  log-softmax — ready for nn.CTCLoss
        """
        # CNN feature extraction
        feat = self.cnn(x)                  # (N, 64, 1, W')

        # Squeeze height → (N, 64, W')  then permute → (N, W', 64)
        feat = feat.squeeze(2)              # (N, 64, W')
        feat = feat.permute(0, 2, 1)        # (N, T, 64)

        # BiLSTM stack
        out, _ = self.lstm1(feat)           # (N, T, 256)
        out, _ = self.lstm2(out)            # (N, T, 1024)
        out, _ = self.lstm3(out)            # (N, T, 1024)
        out, _ = self.lstm4(out)            # (N, T, 1024)
        out, _ = self.lstm5(out)            # (N, T, 256)

        # Dense projection
        out = F.relu(self.dense1(out))      # (N, T, 128)
        out = self.dense2(out)              # (N, T, num_characters)

        # Log-softmax for nn.CTCLoss
        out = F.log_softmax(out, dim=-1)    # (N, T, num_characters)

        # CTCLoss expects (T, N, C)
        out = out.permute(1, 0, 2)          # (T, N, num_characters)
        return out

In [12]:
model = CRNN(NUM_CHARACTERS).to(DEVICE)

criterion = nn.CTCLoss(
    blank=len(ALPHABETS),
    reduction="mean",
    zero_infinity=True
)

optimizer = Adam(
    model.parameters(),
    lr=LR
)

print(model)

CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): SELU()
    (2): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): SELU()
    (5): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): SELU()
    (8): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): SELU()
    (10): Conv2d(128, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): SELU()
    (12): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): SELU()
    (14): Dropout2d(p=0.2, inplace=False)
    (15): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (16): SELU()
    (17): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (18): SELU

In [17]:
def compute_accuracy(log_probs, labels, blank):

    probs = (
        log_probs
        .permute(1, 0, 2)
        .exp()
        .detach()
        .cpu()
        .numpy()
    )

    decoded = ctc_decoder(probs)

    correct = 0

    for pred, target in zip(decoded, labels.cpu().numpy()):

        gt = ""

        for x in target:

            if x == blank:
                break

            gt += ALPHABETS[x]

        if pred == gt:
            correct += 1

    return correct / len(decoded)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, blank):

    model.train()

    total_loss = 0
    total_acc = 0
    n = 0

    print("Entered train_epoch")

    for batch_idx, (images, labels) in enumerate(loader):

        print(f"Batch {batch_idx}")

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        print("Forward")

        log_probs = model(images)

        print("Forward Done")

        T, N, _ = log_probs.shape

        target_lengths = (labels != blank).sum(dim=1).cpu()

        flat_targets = torch.cat([
            labels[i, :tl]
            for i, tl in enumerate(target_lengths)
        ]).cpu()

        input_lengths = torch.full(
            (N,),
            T,
            dtype=torch.long
        )

        print("Loss")

        loss = criterion(
            log_probs,
            flat_targets,
            input_lengths,
            target_lengths
        )

        print("Backward")

        optimizer.zero_grad()
        loss.backward()

        print("Step")

        optimizer.step()

        total_loss += loss.item()

        n += 1

        if batch_idx == 2:
            break

    return total_loss / max(n,1), 0

In [21]:
@torch.no_grad()
def eval_epoch(model, loader, criterion, blank):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        log_probs = model(images)
        T, N, _ = log_probs.shape

        target_lengths = (labels != blank).sum(dim=1).cpu()
        flat_targets   = torch.cat([
            labels[i, :tl] for i, tl in enumerate(target_lengths)
        ]).cpu()
        input_lengths = torch.full((N,), T, dtype=torch.long)

        loss = criterion(log_probs, flat_targets, input_lengths, target_lengths)
        total_loss += loss.item()
        total_acc  += compute_accuracy(log_probs, labels, blank)
        n += 1

    return total_loss / n, total_acc / n

In [22]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

images = images.to(DEVICE)

with torch.no_grad():
    out = model(images)

print(out.shape)

torch.Size([16, 1, 32, 128])
torch.Size([16, 19])
torch.Size([31, 16, 64])


In [23]:
images, labels = next(iter(train_loader))

images = images.to(DEVICE)
labels = labels.to(DEVICE)

log_probs = model(images)

print("forward done")

T, N, _ = log_probs.shape

blank = len(ALPHABETS)

target_lengths = (labels != blank).sum(dim=1).cpu()

print("target lengths done")

flat_targets = torch.cat([
    labels[i, :tl]
    for i, tl in enumerate(target_lengths)
]).cpu()

print("flat targets done")

input_lengths = torch.full(
    (N,),
    T,
    dtype=torch.long
)

print("input lengths done")

loss = criterion(
    log_probs,
    flat_targets,
    input_lengths,
    target_lengths
)

print(loss)

forward done
target lengths done
flat targets done
input lengths done
tensor(4.1818, device='cuda:0', grad_fn=<MeanBackward0>)


In [24]:
print(labels[0])
print(target_lengths[:5])

tensor([44, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63,
        63], device='cuda:0')
tensor([1, 5, 5, 2, 1])
